### Импорт

In [ ]:
#Свалка заметок

#X_train["Тип кузова"].value_counts(dropna=False) #количество каждого уникального значения
#X_train.nunique()
#X_train.describe()
#X_test["Предложение"].isna().sum()
#X_train["Предложение"].duplicated()

#one-hot encoding - приведение категориальных признаков к регрессионному
#fit - метод обучения модели либо енкодера

#ohe_drop = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
#ohe_drop.fit(X_train)
#X_train = ohe_drop.transform(X_train)
#X_train = ohe_drop.fit_transform(X_train)
#X_test = ohe_drop.transform(X_test)

#Лучше не делать ансамбли, а заэнкодить категориальные в регрессионные

#y_train.describe()
#X_train.nunique()
#y_train.nunique()
#X_test.nunique()

#fit не сопоставляет выборки особым образом,
#просто 1строка1выборка = 1строка2выборка; 2строка1выборка = 2строка2выборка

In [ ]:
import zipfile
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostRegressor

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

Задача - Спрогнозировать цену автомобиля на основе набора характеристик. Для обучения предложено 48 наборов данных в разных форматах об автомобилях, отдельно таблица с данными о цене, и тестовая выборка без информации о цене. Для объединения наборов реализована функция, работающая с разными расширениями файлов.

In [ ]:
#Конкатенация файлов с данными, с учетом расширения
def mergingDatas():
    X_train = pd.read_csv('dataset_1/data.csv')

    for i in range(2, 49):
        folder_name = f"./dataset_{i}"
        files = os.listdir(folder_name)
        file_name = files[0]
        _, ext = os.path.splitext(file_name)
        full_path = os.path.join(folder_name, file_name)
        X_i = 0
        if ext == ".csv":
            X_i = pd.read_csv(full_path)
        elif ext == ".json":
            X_i = pd.read_json(full_path)
        elif ext == ".xlsx":
            X_i = pd.read_excel(full_path)
        elif ext == ".parquet":
            X_i = pd.read_parquet(full_path)
        X_train = pd.concat([X_train, X_i], ignore_index=True)
    return X_train

pd.options.display.max_columns = 300
pd.options.display.max_rows = 300
pd.options.display.max_colwidth = 300

#Для X_train используется функция объединения таблиц данных mergingDatas()
X_train = mergingDatas()
y_train = pd.read_csv('y_train_base.csv')
X_test = pd.read_csv('X_test_base.csv')

display( '---Тренировочные данные---',
         X_train.shape,
         X_train.columns,
         X_train.info(),
         '---Тренировочные цены---',
         y_train.shape,
         y_train.columns,
         y_train.info(),
         '---Тестовые данные---',
         X_test.shape,
         X_test.columns,
         X_test.info(),
         '---Соответствие---',
         X_test[X_test['car_id'].isin(y_train['car_id'].unique())].shape
)

X_train

In [ ]:
X_train.describe(include='all')

### Очистка признаков X_train

В полном датасете часто встречаются записи none, null, а также пропуски, записанные как "-". Они заменяются на NaN, чтобы дальнейшая обработка пропусков также учла и их. В столбце "Цвет" также встречаются пропуски "- / -", но их обрабатывать нецелесообразно т.к. "Цвет" не используется в выборке, об этом далее будет уточнено.

Объединённая база данных в большинстве своём состоит из дубликатов. Было сделано предположение, что некоторые дубликаты одной строки могут содержать данные в столбцах, которые не встречаются в других(в т.ч. в первом появлении). Для сохранения этих данных используется заполнение ffill() и bfill() в сгруппированных по car_id подвыборкам. После заполнения пропусков все дубликаты удаляются, остается только первое нахождение.

Для собственного изучения результатов все итоговые выборки дополнительно экспортированы в папку dataset_full.

In [ ]:
X_train = X_train.replace("-", np.nan)
filled = (
    X_train.groupby('car_id')
      .transform(lambda x: x.ffill().bfill())
)
X_train.update(filled)
X_train = X_train.drop_duplicates('car_id', keep='first')


X_train.to_csv('dataset_full/X_train.csv', index=False)
y_train.to_csv('dataset_full/y_train.csv', index=False)
X_test.to_csv('dataset_full/X_test.csv', index=False)

display(
    X_train.info(),
    y_train.info(),
    X_test.info()
)

Каждый параметр был изучен отдельно, были выявлены некоторые специфики (описаны в комментариях). В итоговом датасете остаются только 2 типа: str для категориальных и float64 для численных. Пропуски в категориальных значениях были заменены на моду по столбцу. Пропуски в численных заменены на медиану.

In [ ]:
#Бренд - 66 категорий, предположительно высоко значимых для формирования цены
X_train["Бренд"] = X_train["Бренд"].fillna(X_train["Бренд"].mode()[0])
X_train["Бренд"].value_counts(dropna=False)

In [ ]:
#Год выпуска
X_train["Год выпуска"].value_counts(dropna=False)

In [ ]:
#Модель. Слишком много категорий, которые затруднительно сгруппировать.
#Самая очевидная группировка - по бренду, он уже имеется.
X_train["Модель"] = X_train["Модель"].fillna(X_train["Модель"].mode()[0])
X_train["Модель"].value_counts(dropna=False)

In [ ]:
#Тип машины - очень много категорий, некоторые из которых по смыслу типом машины не являются.
X_train["Тип машины"] = X_train["Тип машины"].fillna(X_train["Тип машины"].mode()[0])
X_train["Тип машины"].value_counts(dropna=False)

In [ ]:
#Полное название - чаще всего собирательная категория из года, бренда, модели и других столбцов.
#Из неё можно было бы получить данные для пропусков в других столбцах, но категории разнородные.
X_train["Полное название"].value_counts(dropna=False)

In [ ]:
#Исползование
X_train["Исползование"].value_counts(dropna=False)

In [ ]:
#КПП
X_train["КПП"] = X_train["КПП"].fillna(X_train["КПП"].mode()[0])
X_train["КПП"].value_counts(dropna=False)

In [ ]:
#Двигатель - содержит информацию о количестве цилиндров и объёме двигателя.
#В двигателе оставлена информацию только об объёме (без единицы измерения)
#Цилиндры уже содержатся в другом параметре
X_train["Двигатель"] = (X_train["Двигатель"].str.split().str[-2])
X_train["Двигатель"] = pd.to_numeric(X_train["Двигатель"],errors="coerce")
X_train["Двигатель"] = X_train["Двигатель"].fillna(X_train["Двигатель"].mode()[0])
X_train["Двигатель"] = X_train["Двигатель"].astype(float)
X_train["Двигатель"].value_counts(dropna=False)

In [ ]:
#Привод
X_train["Привод"].value_counts(dropna=False)

In [ ]:
#Топливо
X_train["Топливо"] = X_train["Топливо"].fillna(X_train["Топливо"].mode()[0])
X_train["Топливо"].value_counts(dropna=False)

In [ ]:
#Расход - убрана единица измерения, осталось только число
X_train["Расход"] = (X_train["Расход"].str.split().str[0])
X_train["Расход"] = pd.to_numeric(X_train["Расход"],errors="coerce")
X_train["Расход"] = X_train["Расход"].fillna(X_train["Расход"].median())
X_train["Расход"] = X_train["Расход"].astype(float)
X_train["Расход"].value_counts(dropna=False)

In [ ]:
#Пробег - некоторые значения напоминают не пробег, а цвет.
#В той же строке в параметре "Цвет" присутствует информация о пробеге
#После перемены значений местами с помощью regex устраняются ед. измерения
mask_swap = X_train["Пробег"].astype(str).str.contains(r"-\s*/\s*-", na=False)
X_train.loc[mask_swap, ["Пробег", "Цвет"]] = X_train.loc[mask_swap, ["Цвет", "Пробег"]].values

import re
def extract_mileage(x):
    nums = re.findall(r"\d[\d,]*", str(x))
    if not nums:
        return None
    return int(nums[-1].replace(",", ""))

X_train["Пробег"] = X_train["Пробег"].apply(extract_mileage)
X_train["Пробег"] = X_train["Пробег"].fillna(X_train["Пробег"].median())
X_train["Пробег"] = X_train["Пробег"].astype(float)
X_train["Пробег"].value_counts(dropna=False)

In [ ]:
#Цвет - Много категорий, сложно упрощаемых
X_train["Цвет"].value_counts(dropna=False)

In [ ]:
#Локация - очень много категорий (572 rows), которые можно группировать по штатам
#С помощью regex выделены штаты в концах значений, итоговое количество более приемлемо
X_train["Локация"] = (X_train["Локация"].str.extract(r'([A-Z]{2,3})$', expand=False)
                      .fillna(X_train["Локация"].str.extract(r'([A-Z]{2,3})$', expand=False).mode()[0]))
X_train["Локация"].value_counts(dropna=False)

In [ ]:
#Количество цилиндров - убрана приписка "cyl"
X_train["Количество цилиндров"] = (X_train["Количество цилиндров"].str.split().str[0])
X_train["Количество цилиндров"] = X_train["Количество цилиндров"].fillna(X_train["Количество цилиндров"].mode()[0])
X_train["Количество цилиндров"] = X_train["Количество цилиндров"].astype(float)
X_train["Количество цилиндров"].value_counts(dropna=False)

In [ ]:
#Тип кузова - применимый категориальный признак
X_train["Тип кузова"] = (X_train["Тип кузова"].str.split().str[0])
X_train["Тип кузова"] = X_train["Тип кузова"].fillna(X_train["Тип кузова"].mode()[0])
X_train["Тип кузова"] = X_train["Тип кузова"].astype(str)
X_train["Тип кузова"].value_counts(dropna=False)

In [ ]:
#Двери - убрана приписка "doors", оставлено число - количество
X_train["Двери"] = (X_train["Двери"].str.split().str[0])
X_train["Двери"] = pd.to_numeric(X_train["Двери"],errors="coerce")
X_train["Двери"] = X_train["Двери"].fillna(X_train["Двери"].mode()[0])
X_train["Двери"] = X_train["Двери"].astype(float)
X_train["Двери"].value_counts(dropna=False)

In [ ]:
#Количество кресел - убрана приписка "seats", оставлено число - количество
X_train["Количество кресел"] = (X_train["Количество кресел"].str.split().str[0])
X_train["Количество кресел"] = pd.to_numeric(X_train["Количество кресел"],errors="coerce")
X_train["Количество кресел"] = X_train["Количество кресел"].fillna(X_train["Количество кресел"].mode()[0])
X_train["Количество кресел"] = X_train["Количество кресел"].astype(float)
X_train["Количество кресел"].value_counts(dropna=False)

In [ ]:
#Оценка эксперта
X_train["Оценка эксперта"] = X_train["Оценка эксперта"].astype(float)
X_train["Оценка эксперта"].value_counts(dropna=False)

In [ ]:
#Количество владельцев
X_train["Количество владельцев"] = X_train["Количество владельцев"].astype(float)
X_train["Количество владельцев"].value_counts(dropna=False)

In [ ]:
#Предложение
X_train["Предложение"] = X_train["Предложение"].astype(float)
X_train["Предложение"].value_counts(dropna=False)

In [ ]:
X_train.info()

In [ ]:
X_train.describe(include='all')

### Очистка признаков X_test

In [ ]:
#Функция очистки для X_test, повторяет все ранее описанные действия, но без вывода .value_counts()
X_test = pd.read_csv('X_test_base.csv')
def X_test_filter(X_train):
    X_train = X_train.replace("-", np.nan)
    X_train["Бренд"] = X_train["Бренд"].fillna(X_train["Бренд"].mode()[0])

    X_train["Модель"] = X_train["Модель"].fillna(X_train["Модель"].mode()[0])

    X_train["Тип машины"] = X_train["Тип машины"].fillna(X_train["Тип машины"].mode()[0])

    X_train["КПП"] = X_train["КПП"].fillna(X_train["КПП"].mode()[0])

    X_train["Двигатель"] = (X_train["Двигатель"].str.split().str[-2])
    X_train["Двигатель"] = pd.to_numeric(X_train["Двигатель"],errors="coerce")
    X_train["Двигатель"] = X_train["Двигатель"].fillna(X_train["Двигатель"].mode()[0])
    X_train["Двигатель"] = X_train["Двигатель"].astype(float)

    X_train["Топливо"] = X_train["Топливо"].fillna(X_train["Топливо"].mode()[0])

    X_train["Расход"] = (X_train["Расход"].str.split().str[0])
    X_train["Расход"] = pd.to_numeric(X_train["Расход"],errors="coerce")
    X_train["Расход"] = X_train["Расход"].fillna(X_train["Расход"].median())
    X_train["Расход"] = X_train["Расход"].astype(float)

    mask_swap = X_train["Пробег"].astype(str).str.contains(r"-\s*/\s*-", na=False)
    X_train.loc[mask_swap, ["Пробег", "Цвет"]] = X_train.loc[mask_swap, ["Цвет", "Пробег"]].values
    X_train["Пробег"] = X_train["Пробег"].apply(extract_mileage)
    X_train["Пробег"] = X_train["Пробег"].fillna(X_train["Пробег"].median())
    X_train["Пробег"] = X_train["Пробег"].astype(float)

    X_train["Локация"] = (X_train["Локация"].str.extract(r'([A-Z]{2,3})$', expand=False)
        .fillna(X_train["Локация"].str.extract(r'([A-Z]{2,3})$', expand=False).mode()[0]))

    X_train["Количество цилиндров"] = (X_train["Количество цилиндров"].str.split().str[0])
    X_train["Количество цилиндров"] = X_train["Количество цилиндров"].fillna(X_train["Количество цилиндров"].mode()[0])
    X_train["Количество цилиндров"] = X_train["Количество цилиндров"].astype(float)

    X_train["Тип кузова"] = (X_train["Тип кузова"].str.split().str[0])
    X_train["Тип кузова"] = X_train["Тип кузова"].fillna(X_train["Тип кузова"].mode()[0])
    X_train["Тип кузова"] = X_train["Тип кузова"].astype(str)

    X_train["Двери"] = (X_train["Двери"].str.split().str[0])
    X_train["Двери"] = pd.to_numeric(X_train["Двери"],errors="coerce")
    X_train["Двери"] = X_train["Двери"].fillna(X_train["Двери"].mode()[0])
    X_train["Двери"] = X_train["Двери"].astype(float)

    X_train["Количество кресел"] = (X_train["Количество кресел"].str.split().str[0])
    X_train["Количество кресел"] = pd.to_numeric(X_train["Количество кресел"],errors="coerce")
    X_train["Количество кресел"] = X_train["Количество кресел"].fillna(X_train["Количество кресел"].mode()[0])
    X_train["Количество кресел"] = X_train["Количество кресел"].astype(float)

    X_train["Оценка эксперта"] = X_train["Оценка эксперта"].astype(float)

    X_train["Количество владельцев"] = X_train["Количество владельцев"].astype(float)
    return X_train
X_test = X_test_filter(X_test)

### Формирование выборок для обучения

In [ ]:
#Объединение X_train и y_train для состыковки по строкам

print(X_train.shape)
print(y_train.shape)

X_train_with_price = X_train.merge(y_train, on='car_id')

print("Итоговая выборка с ценами")
print(X_train_with_price.shape)

Для определения минимального набора необходимых параметров была составлена матрица, описывающая корреляцию между числовыми признаками. В результате было выявлено, что высокую корреляцию имеют:
- "Двигатель" и "Количество цилиндров"
- "Предложение" и "Цена"

Эти параметры далее не будут использоваться в обучении. Кроме того, параметр "Предложение" не имеет данных в тестовой выборке, обученная на этом параметре модель покажет плохие результаты, если данные будут отсутствовать.

Остальные параметры необходимо изучить отдельно.

In [ ]:
X_train_with_price_corr = X_train_with_price.select_dtypes('number')
plt.figure(figsize=(10, 4))

sns.set(font_scale=1)

corr_matrix = X_train_with_price_corr.corr()
corr_matrix = np.round(corr_matrix, 2)

sns.heatmap(corr_matrix, annot=True, linewidths=.5, cmap='coolwarm')

plt.title('Correlation matrix')
plt.show()

Методом подбора был выявлен набор признаков, дающий наибольшее качество при обучении. В списках features и features_categorical перечислены признаки, в дальнейшем применяемые для обучения

In [ ]:
features = ["Год выпуска","Двигатель","Расход","Пробег"]
features_categorical = ["Бренд","Исползование","КПП","Привод","Топливо","Локация","Тип кузова"]
trash = [
    #Параметры, которые не будут использоваться
    "Модель","Тип машины","Полное название","Цвет","Предложение",

    #Параметры, на которых модели не показали прироста качества(а чаще и наоборот)
    "Количество цилиндров","Двери","Количество кресел","Оценка эксперта","Количество владельцев"
]

#Сформированы наборы входных и выходного параметра
X = X_train_with_price[features + features_categorical]
y = X_train_with_price['Цена']

display(
    X.info(),
    y.info())

In [ ]:
#Разбиение наборов на обучающие и валидационные
#Валидационные наборы необходимы для определения качества модели
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X))
print(len(y))

Для разбиения категориальных признаков на числовые используется метод One Hot Encoder. В дальнейшем будут некоторые рассмотрены модели, для которых это разбиение необходимо. В результате образуются датасеты с неизмененными числовыми и преобразованными категориальными признаками.

In [ ]:
ohe_drop = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')

ohe_drop.fit(X_train[features_categorical])

X_train_cat = ohe_drop.transform(X_train[features_categorical])
X_valid_cat = ohe_drop.transform(X_valid[features_categorical])
X_test_cat  = ohe_drop.transform(X_test[features_categorical])

X_train_num = X_train[features]
X_valid_num = X_valid[features]
X_test_num  = X_test[features]

X_train_final = np.hstack([X_train_num, X_train_cat])
X_valid_final = np.hstack([X_valid_num, X_valid_cat])
X_test_final  = np.hstack([X_test_num, X_test_cat])

## Построение моделей

Были рассмотрены несколько моделей:
- Линейная регрессия
- Дерево решений
- Случайный лес
- Градиентный бустинг (CatBoost)

Гиперпараметры моделей "Дерево решений" и "Случайный лес" были определены эмпирически.
Для модели CatBoost были аналогичным образом определены оптимальные гиперпараметры, после чего был использован метод RandomisedSearchCV, случайно подбирающий гиперпараметры из определенных списков и выделяющий набор, давший модели наибольшую точность. Самая лучшая модель была вынесена в отдельную часть.
Для оценки качества работы модели были использованы следующие метрики:
- MSE - средняя квадратичная ошибка
- RMSE - корень из MSE
- MAE - средняя абсолютная ошибка
- R² - коэффициент детерминации

### Линейная регрессия

In [ ]:
#Линейная регрессия
linear = LinearRegression()
linear.fit(X_train_final, y_train)

pred_linear = linear.predict(X_valid_final)

print(f'MSE: {mean_squared_error(y_valid, pred_linear):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_valid, pred_linear)):.2f}')
print(f'MAE: {mean_absolute_error(y_valid, pred_linear):.2f}')
print(f'R²: {r2_score(y_valid, pred_linear):.6f}')

y_test_linear = linear.predict(X_test_final)
y_test_linear = pd.Series(y_test_linear, name='Цена')

### Дерево решений

In [ ]:
#Лучший набор гиперпараметров: max_depth=27, min_samples_leaf=5
regtree = DecisionTreeRegressor(max_depth=27, min_samples_leaf=5)
regtree.fit(X_train_final, y_train)
pred_y_tree = regtree.predict(X_valid_final)

print(f'MSE: {mean_squared_error(y_valid, pred_y_tree):.2f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_valid, pred_y_tree)):.2f}')
print(f'MAE: {mean_absolute_error(y_valid, pred_y_tree):.2f}')
print(f'R²: {r2_score(y_valid, pred_y_tree):.6f}')

y_test_tree = regtree.predict(X_test_final)
y_test_tree = pd.Series(y_test_tree, name='Цена')

### Случайный лес

In [ ]:
#Лучшие: n_estimators= 199, bootstrap=True, max_features=0.7
rf = RandomForestRegressor(n_estimators= 199, bootstrap=True, max_features=0.7, random_state=42, n_jobs=-1)
rf.fit(X_train_final, y_train)
pred_rf = rf.predict(X_valid_final)

print(f"MSE : {mean_squared_error(y_valid, pred_rf):.2f}")
print(f"RMSE: {mean_squared_error(y_valid, pred_rf) ** 0.5:.2f}")
print(f"MAE: {mean_absolute_error(y_valid, pred_rf):.2f}")
print(f"R²  : {r2_score(y_valid, pred_rf):.6f}")

y_test_rf = rf.predict(X_test_final)
y_test_rf = pd.Series(y_test_rf, name='Цена')

### CatBoost

In [ ]:
#Пробная реализация CatBoost
cat_features = [X_train.columns.get_loc(col) for col in features_categorical]

cat = CatBoostRegressor(iterations=1000, learning_rate=0.1, l2_leaf_reg=1, depth=7, random_seed=42, verbose=False)
cat.fit(X_train, y_train, cat_features=cat_features)
pred_cat = cat.predict(X_valid)

print(f"MSE : {mean_squared_error(y_valid, pred_cat):.2f}")
print(f"RMSE: {mean_squared_error(y_valid, pred_cat) ** 0.5:.2f}")
print(f"MAE: {mean_absolute_error(y_valid, pred_cat):.2f}")
print(f"R²  : {r2_score(y_valid, pred_cat):.6f}")

y_test_cat = cat.predict(X_test[features + features_categorical])
y_test_cat = pd.Series(y_test_cat, name='Цена')

### CatBoost Randomized Best

In [ ]:
#аналогичная реализация, но с наилучшими гиперпараметрами, найденными через RandomisedSearchCV
cat_features = [X_train.columns.get_loc(col) for col in features_categorical]

cat = CatBoostRegressor(iterations=1500, learning_rate=0.1, l2_leaf_reg=1, depth=8, random_seed=42, verbose=False)
cat.fit(X_train, y_train, cat_features=cat_features)
pred_cat = cat.predict(X_valid)

print(f"MSE : {mean_squared_error(y_valid, pred_cat):.2f}")
print(f"RMSE: {mean_squared_error(y_valid, pred_cat) ** 0.5:.2f}")
print(f"MAE: {mean_absolute_error(y_valid, pred_cat):.2f}")
print(f"R²  : {r2_score(y_valid, pred_cat):.6f}")

y_test_cat = cat.predict(X_test[features + features_categorical])
y_test_cat = pd.Series(y_test_cat, name='Цена')

cat.save_model("catboost_model.cbm")

y_test_cat.to_csv('submission.csv', index=False)
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('submission.csv')

### CatBoost Randomized

In [ ]:
#Подбор наилучших параметров
cat = CatBoostRegressor(verbose=False,random_seed=42)

params = {
    "iterations": [500,1000,1500],
    "depth": [4,5,6,7,8],
    "learning_rate": [0.01,0.03,0.05,0.1],
    "l2_leaf_reg": [1,3,5,7,10]
}

search = RandomizedSearchCV(cat,params,n_iter=50,cv=3,scoring="neg_root_mean_squared_error",n_jobs=-1)
search.fit(X_train,y_train,cat_features=features_categorical)

print(search.best_params_)
best_cat = search.best_estimator_
pred_cat = best_cat.predict(X_valid)

print(f"MSE : {mean_squared_error(y_valid, pred_cat):.2f}")
print(f"RMSE: {mean_squared_error(y_valid, pred_cat) ** 0.5:.2f}")
print(f"MAE: {mean_absolute_error(y_valid, pred_cat):.2f}")
print(f"R²  : {r2_score(y_valid, pred_cat):.6f}")


y_test_cat = best_cat.predict(X_test[features + features_categorical])
y_test_cat = pd.Series(y_test_cat, name='Цена')

### Итог
Итоговые метрики показали, что модель CatBoost имеет самое наибольшее качество по сравнению с другими рассмотренными моделями МО.
Из результатов прогноза этой модели на тестовой выборке был сформирован файл submission.csv